In [1]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Dense,
    Dropout,
    GlobalAveragePooling2D
)


base_model = tf.keras.applications.MobileNetV2(
    input_shape=(96,96,3),
    include_top=False,
    weights="imagenet"
)


base_model.trainable = False


x = base_model.output

x = GlobalAveragePooling2D()(x)

x = Dense(
    256,
    activation="relu"
)(x)

x = Dropout(0.4)(x)


output = Dense(
    7,
    activation="softmax"
)(x)


model = Model(
    inputs=base_model.input,
    outputs=output
)


model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.0001
    ),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)


model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 96, 96, 3)]          0         []                            
                                                                                                  
 Conv1 (Conv2D)              (None, 48, 48, 32)           864       ['input_1[0][0]']             
                                                                                                  
 bn_Conv1 (BatchNormalizati  (None, 48, 48, 32)           128       ['Conv1[0][0]']               
 on)                                                                                              
                                                                                                  
 Conv1_relu (ReLU)           (None, 48, 48, 32)           0         ['bn_Conv1[0][0]']        

In [2]:
IMG_SIZE = (96,96)

In [3]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator


IMG_SIZE = (96,96)


datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,

    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True
)

In [10]:
import os

for root, dirs, files in os.walk("../datasets"):
    level = root.count(os.sep)
    if level < 4:
        print(root, "->", len(files), "files")

../datasets -> 0 files
../datasets\facial -> 0 files
../datasets\facial\test -> 0 files
../datasets\facial\test\angry -> 958 files
../datasets\facial\test\disgust -> 111 files
../datasets\facial\test\fear -> 1024 files
../datasets\facial\test\happy -> 1774 files
../datasets\facial\test\neutral -> 1233 files
../datasets\facial\test\sad -> 1247 files
../datasets\facial\test\surprise -> 831 files
../datasets\facial\train -> 0 files
../datasets\facial\train\angry -> 3995 files
../datasets\facial\train\disgust -> 436 files
../datasets\facial\train\fear -> 4097 files
../datasets\facial\train\happy -> 7215 files
../datasets\facial\train\neutral -> 4965 files
../datasets\facial\train\sad -> 4830 files
../datasets\facial\train\surprise -> 3171 files
../datasets\facial_emotion -> 0 files
../datasets\facial_emotion\train -> 0 files
../datasets\facial_emotion\val -> 0 files
../datasets\questionnaire -> 1 files
../datasets\speech -> 0 files
../datasets\speech\Audio_Speech_Actors_01-24 -> 0 files
..

In [4]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam


base_model = tf.keras.applications.MobileNetV2(
    input_shape=(96,96,3),
    include_top=False,
    weights="imagenet"
)


# freeze pretrained layers first
base_model.trainable = False


x = base_model.output

x = GlobalAveragePooling2D()(x)

x = Dense(
    256,
    activation="relu"
)(x)

x = Dropout(0.4)(x)


output = Dense(
    7,
    activation="softmax"
)(x)


model = Model(
    inputs=base_model.input,
    outputs=output
)


model.compile(
    optimizer=Adam(
        learning_rate=0.0001
    ),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)


model.summary()

Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_2 (InputLayer)        [(None, 96, 96, 3)]          0         []                            
                                                                                                  
 Conv1 (Conv2D)              (None, 48, 48, 32)           864       ['input_2[0][0]']             
                                                                                                  
 bn_Conv1 (BatchNormalizati  (None, 48, 48, 32)           128       ['Conv1[0][0]']               
 on)                                                                                              
                                                                                                  
 Conv1_relu (ReLU)           (None, 48, 48, 32)           0         ['bn_Conv1[0][0]']      

In [5]:
model.summary()

Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_2 (InputLayer)        [(None, 96, 96, 3)]          0         []                            
                                                                                                  
 Conv1 (Conv2D)              (None, 48, 48, 32)           864       ['input_2[0][0]']             
                                                                                                  
 bn_Conv1 (BatchNormalizati  (None, 48, 48, 32)           128       ['Conv1[0][0]']               
 on)                                                                                              
                                                                                                  
 Conv1_relu (ReLU)           (None, 48, 48, 32)           0         ['bn_Conv1[0][0]']      

In [6]:
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)


checkpoint = ModelCheckpoint(
    "facial_mobilenet_best.h5",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max"
)


early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)


reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    patience=3,
    factor=0.2,
    verbose=1
)

In [26]:
del checkpoint

In [7]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator


IMG_SIZE = (96,96)


train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)


test_datagen = ImageDataGenerator(
    rescale=1./255
)


train_generator = train_datagen.flow_from_directory(
    "../datasets/facial/train",
    target_size=IMG_SIZE,
    batch_size=32,
    color_mode="rgb",
    class_mode="categorical"
)


validation_generator = test_datagen.flow_from_directory(
    "../datasets/facial/test",
    target_size=IMG_SIZE,
    batch_size=32,
    color_mode="rgb",
    class_mode="categorical"
)

Found 28709 images belonging to 7 classes.
Found 7178 images belonging to 7 classes.


In [33]:
print(train_generator.samples)
print(validation_generator.samples)

28709
7178


In [8]:
history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=20,
    callbacks=[
        checkpoint,
        early_stopping,
        reduce_lr
    ]
)

Epoch 1/20
898/898 [==============================] - ETA: 0s - loss: 1.7471 - accuracy: 0.3193

c:\Users\Admin\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


898/898 [==============================] - 125s 137ms/step - loss: 1.7471 - accuracy: 0.3193 - val_loss: 1.5632 - val_accuracy: 0.4044 - lr: 1.0000e-04
Epoch 2/20
898/898 [==============================] - 121s 135ms/step - loss: 1.5909 - accuracy: 0.3806 - val_loss: 1.5186 - val_accuracy: 0.4143 - lr: 1.0000e-04
Epoch 3/20
898/898 [==============================] - 127s 141ms/step - loss: 1.5535 - accuracy: 0.3987 - val_loss: 1.5091 - val_accuracy: 0.4259 - lr: 1.0000e-04
Epoch 4/20
898/898 [==============================] - 123s 136ms/step - loss: 1.5245 - accuracy: 0.4112 - val_loss: 1.4887 - val_accuracy: 0.4294 - lr: 1.0000e-04
Epoch 5/20
898/898 [==============================] - 135s 150ms/step - loss: 1.5039 - accuracy: 0.4154 - val_loss: 1.4819 - val_accuracy: 0.4352 - lr: 1.0000e-04
Epoch 6/20
898/898 [==============================] - 141s 157ms/step - loss: 1.4856 - accuracy: 0.4276 - val_loss: 1.4752 - val_accuracy: 0.4368 - lr: 1.0000e-04
Epoch 7/20
898/898 [=============

In [9]:
base_model.trainable = True

# freeze only early layers
for layer in base_model.layers[:-50]:
    layer.trainable = False


model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-5
    ),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [10]:
history_ft = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=15,
    callbacks=[
        checkpoint,
        early_stopping,
        reduce_lr
    ]
)

Epoch 1/15
898/898 [==============================] - 165s 179ms/step - loss: 1.6876 - accuracy: 0.3557 - val_loss: 1.4837 - val_accuracy: 0.4308 - lr: 1.0000e-05
Epoch 2/15
898/898 [==============================] - 160s 179ms/step - loss: 1.5128 - accuracy: 0.4165 - val_loss: 1.4326 - val_accuracy: 0.4507 - lr: 1.0000e-05
Epoch 3/15
898/898 [==============================] - 158s 176ms/step - loss: 1.4489 - accuracy: 0.4456 - val_loss: 1.3966 - val_accuracy: 0.4659 - lr: 1.0000e-05
Epoch 4/15
898/898 [==============================] - 164s 182ms/step - loss: 1.4056 - accuracy: 0.4620 - val_loss: 1.3715 - val_accuracy: 0.4724 - lr: 1.0000e-05
Epoch 5/15
898/898 [==============================] - 164s 183ms/step - loss: 1.3656 - accuracy: 0.4772 - val_loss: 1.3445 - val_accuracy: 0.4859 - lr: 1.0000e-05
Epoch 6/15
898/898 [==============================] - 164s 183ms/step - loss: 1.3384 - accuracy: 0.4901 - val_loss: 1.3346 - val_accuracy: 0.4898 - lr: 1.0000e-05
Epoch 7/15
898/898 [==

In [11]:
test_loss, test_acc = model.evaluate(
    validation_generator
)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_acc)

225/225 [==============================] - 20s 88ms/step - loss: 1.2259 - accuracy: 0.5407
Test Loss: 1.2258656024932861
Test Accuracy: 0.5406798720359802


In [12]:
model.save(
    "../models/saved_models/facial_emotion_mobilenet.keras"
)